# Regression. Exploratory model selection   

## We first look at what our previous analysis revealed

In [5]:
import statsmodels.api as sm
import pandas as pd

df_processed = pd.read_csv("../data/processed/heart_failure_clinical_records_dataset_processed.csv")
# Manual Model
features = [
    "ejection_creatinine_interaction",
    "sodium_creatinine_interaction",
    "serum_sodium"
]
X = df_processed[features]
y = df_processed["ejection_fraction_centered"]
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

                                OLS Regression Results                                
Dep. Variable:     ejection_fraction_centered   R-squared:                       0.283
Model:                                    OLS   Adj. R-squared:                  0.276
Method:                         Least Squares   F-statistic:                     38.83
Date:                        Tue, 17 Mar 2026   Prob (F-statistic):           3.53e-21
Time:                                16:56:40   Log-Likelihood:                -1112.8
No. Observations:                         299   AIC:                             2234.
Df Residuals:                             295   BIC:                             2249.
Df Model:                                   3                                         
Covariance Type:                    nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
------------------------------

The regression results indicate that interaction terms between clinical
variables provide the strongest explanatory power for the target variable.

Both interaction predictors are highly statistically significant, while the
independent effect of serum sodium becomes insignificant once these
interactions are included.

The model explains approximately 28% of the variance in the target variable,
which is consistent with the relatively weak correlations observed during
exploratory analysis.

Overall, the results suggest that combinations of clinical variables capture
more information about cardiac function than individual predictors alone.

In [6]:
# Dropping the serum_sodium variable and refitting the model
features = [
    "ejection_creatinine_interaction",
    "sodium_creatinine_interaction"
]
X = df_processed[features]
y = df_processed["ejection_fraction_centered"]
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

                                OLS Regression Results                                
Dep. Variable:     ejection_fraction_centered   R-squared:                       0.280
Model:                                    OLS   Adj. R-squared:                  0.275
Method:                         Least Squares   F-statistic:                     57.58
Date:                        Tue, 17 Mar 2026   Prob (F-statistic):           7.54e-22
Time:                                16:56:44   Log-Likelihood:                -1113.5
No. Observations:                         299   AIC:                             2233.
Df Residuals:                             296   BIC:                             2244.
Df Model:                                   2                                         
Covariance Type:                    nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
------------------------------

## We see from another perspective: Stepwise selection?

In [7]:
import sys
sys.path.append("..")
from src.models.regression.stepwise_selection import run_stepwise_selection_processed

In [12]:
results = run_stepwise_selection_processed(df_processed)

display(results["comparison_table"])
display(results["forward"]["history_table"])
display(results["backward"]["history_table"])

print(results["forward"]["model"].summary())
print(results["backward"]["model"].summary())

,method,n_features,selected_features,aic,bic,r_squared,adj_r_squared
0,forward_aic,5,"ejection_creatinine_interaction, creatinine_lo...",2145.561433,2167.764094,0.473216,0.464227
1,backward_aic,5,"age_centered, creatinine_log, ejection_creatin...",2145.561433,2167.764094,0.473216,0.464227


,step,action,feature,n_features,aic,bic,r_squared,adj_r_squared
0,1,add,ejection_creatinine_interaction,1,2290.870757,2298.271644,0.120345,0.117383
1,2,add,creatinine_log,2,2185.787649,2196.888980,0.385141,0.380987
2,3,add,sodium_creatinine_interaction,3,2149.390624,2164.192398,0.459241,0.453741
3,4,add,age_centered,4,2146.171599,2164.673817,0.468598,0.461368
4,5,add,serum_sodium,5,2145.561433,2167.764094,0.473216,0.464227


,step,action,feature,n_features,aic,bic,r_squared,adj_r_squared
0,1,remove,age_diabetes_interaction,6,2147.535722,2173.438827,0.473262,0.462438
1,2,remove,time,5,2145.561433,2167.764094,0.473216,0.464227


                                OLS Regression Results                                
Dep. Variable:     ejection_fraction_centered   R-squared:                       0.473
Model:                                    OLS   Adj. R-squared:                  0.464
Method:                         Least Squares   F-statistic:                     52.64
Date:                        Mon, 16 Mar 2026   Prob (F-statistic):           7.37e-39
Time:                                01:35:51   Log-Likelihood:                -1066.8
No. Observations:                         299   AIC:                             2146.
Df Residuals:                             293   BIC:                             2168.
Df Model:                                   5                                         
Covariance Type:                    nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
------------------------------

Forward and backward stepwise selection based on AIC were applied to
identify an appropriate regression specification.

Both procedures converged to the same model containing five predictors,
suggesting a stable model structure.

The largest improvement in explanatory power occurred when adding the
log-transformed creatinine variable, highlighting the importance of
feature transformations.

However, most of the predictive power is captured by the first three
predictors, with subsequent variables providing only marginal gains in
model fit.

For this reason, both a parsimonious three-variable model and the
stepwise-selected model are considered in the subsequent analysis.